# TOP-10 ФП/СФП по интервальной логике до дефолта

## Бизнес-логика (взаимоисключающие интервалы)
Для каждого дефолтного ИНН и сегмента считаем ТОП-10 факторов в отдельных интервалах до `default_date`:

- `24_18m`: от 2 лет до 1.5 лет до дефолта
- `18_12m`: от 1.5 года до 1 года
- `12_6m`: от 1 года до полугода
- `6_0m`: за полгода до дефолта

Важно: интервалы взаимоисключающие, фактор попадает только в один интервал по своей `IDENTIFICATION_DATE`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

DATA_DIR = Path("/home/jovyan/documents/DMUKP_FP_DEF/data")
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

VAL4_MAP = {
    "Микро": "МкБ",
    "Малый": "МСБ",
    "Средний": "МСБ",
    "Крупный": "ККБ",
    "Крупнейший": "ККБ",
    "Не подл. сегм.": np.nan,
}

INTERVALS = [
    ("24_18m", 24, 18),
    ("18_12m", 18, 12),
    ("12_6m", 12, 6),
    ("6_0m", 6, 0),
]


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


def mode_safe(series):
    m = series.dropna().mode()
    return m.iloc[0] if len(m) else np.nan

In [ ]:
# Загрузка данных
crm_path = DATA_DIR / "crm_last_version.csv"
def_path = DATA_DIR / "default_data.pkl"

df_crm = pd.read_csv(crm_path, dtype=str, low_memory=False)
df_crm.columns = df_crm.columns.str.strip()
df_crm = df_crm.rename(columns={
    "VAL.1": "VAL_1", "DESC_TEXT.1": "DESC_TEXT_1", "ROW_ID.1": "ROW_ID_1",
    "AGREEMENT_NUM.1": "AGREEMENT_NUM_1", "ROW_ID.2": "ROW_ID_2",
    "VAL.2": "VAL_2", "NAME.1": "NAME_1", "VAL.3": "VAL_3", "VAL.5": "VAL_5",
})

df_def = pd.read_pickle(def_path)
df_def = df_def.astype(str).replace("nan", np.nan)
df_def.columns = df_def.columns.str.strip()
df_def = df_def.rename(columns={"start": "start_date", "cure": "cure_date", "finish": "finish_date"})

print(f"CRM: {len(df_crm):,} строк")
print(f"DEFAULTS: {len(df_def):,} строк")

In [ ]:
# Подготовка defaults (default_date)
severity_priority = {"тяжёлый": 0, "активный": 1, "надзор": 2, "вышел из дефолта": 3}

def classify_severity(row):
    if row.get("unlimited_default") == 1:
        return "тяжёлый"
    if pd.notna(row.get("writeoff")) or pd.notna(row.get("liquidation")):
        return "тяжёлый"
    if pd.notna(row.get("finish_date")):
        return "вышел из дефолта"
    if pd.notna(row.get("cure_date")):
        return "надзор"
    return "активный"

df_def_work = df_def.copy()
df_def_work["inn"] = df_def_work["inn"].apply(normalize_inn)
for col in ["start_date", "cure_date", "finish_date", "writeoff", "liquidation"]:
    if col in df_def_work.columns:
        df_def_work[col] = pd.to_datetime(df_def_work[col], dayfirst=True, errors="coerce")

if "unlimited_default" in df_def_work.columns:
    df_def_work["unlimited_default"] = pd.to_numeric(df_def_work["unlimited_default"], errors="coerce").fillna(0).astype(int)
else:
    df_def_work["unlimited_default"] = 0

df_def_work["severity"] = df_def_work.apply(classify_severity, axis=1)
df_def_work["_sev_rank"] = df_def_work["severity"].map(severity_priority)

defaults = (
    df_def_work
    .dropna(subset=["inn", "start_date"])
    .query("@DATE_FROM <= start_date <= @DATE_TO")
    .sort_values("_sev_rank")
    .groupby("inn", as_index=False)
    .agg(
        default_date=("start_date", "min"),
        severity=("severity", "first"),
    )
)
defaults["default_flag"] = 1

print(f"Дефолтных ИНН в когорте: {len(defaults):,}")
print(f"Период default_date: {defaults['default_date'].min()} — {defaults['default_date'].max()}")

In [ ]:
# Подготовка CRM + сегментация (X_AREA_RESP с fallback на VAL.4)
df_crm_work = df_crm.copy()
df_crm_work["IDENTIFICATION_DATE"] = pd.to_datetime(df_crm_work["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
df_crm_work["X_INN"] = df_crm_work["X_INN"].apply(normalize_inn)

df_crm_work = df_crm_work[
    (df_crm_work["IDENTIFICATION_DATE"] >= DATE_FROM) &
    (df_crm_work["IDENTIFICATION_DATE"] <= DATE_TO)
].copy()

df_crm_work["VAL"] = df_crm_work["VAL"].astype(str).str.strip()
df_crm_work = df_crm_work[df_crm_work["VAL"].isin(ALLOWED_SOURCES)].copy()

df_crm_work["TYPE_FP"] = df_crm_work["TYPE_FP"].astype(str).str.strip().replace({"FP": "ФП", "SFP": "СФП"})
df_crm_work["X_AREA_RESP"] = df_crm_work["X_AREA_RESP"].astype(str).str.strip()
df_crm_work["segment_zo"] = df_crm_work["X_AREA_RESP"].map(SEGMENT_MAP)

val4_col = "VAL.4" if "VAL.4" in df_crm_work.columns else ("VAL_4" if "VAL_4" in df_crm_work.columns else None)
if val4_col is None:
    raise KeyError("Не найдена колонка 'VAL.4'/'VAL_4' в CRM")

df_crm_work["val4_raw"] = df_crm_work[val4_col].astype(str).str.strip()
df_crm_work["segment_val4"] = df_crm_work["val4_raw"].map(VAL4_MAP)

df_crm_work["fp_num"] = df_crm_work["NUMBER_FP_SFP"].astype(str).str.strip()
df_crm_work = df_crm_work.dropna(subset=["X_INN", "NUMBER_FP_SFP"]).copy()
df_crm_work = df_crm_work.drop_duplicates(subset=["X_INN", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()

fp_raw = (
    df_crm_work[["X_INN", "NUMBER_FP_SFP", "TYPE_FP", "IDENTIFICATION_DATE", "segment_zo", "segment_val4"]]
    .rename(columns={
        "X_INN": "inn",
        "NUMBER_FP_SFP": "fp_sfp_code",
        "TYPE_FP": "type_fp",
        "IDENTIFICATION_DATE": "fp_date",
    })
    .dropna(subset=["inn", "fp_sfp_code", "fp_date"])
)

seg_zo = (
    fp_raw[["inn", "segment_zo"]]
    .dropna(subset=["segment_zo"])
    .groupby("inn", as_index=False)
    .agg(segment_zo=("segment_zo", mode_safe))
)
seg_val4 = (
    fp_raw[["inn", "segment_val4"]]
    .dropna(subset=["segment_val4"])
    .groupby("inn", as_index=False)
    .agg(segment_val4=("segment_val4", mode_safe))
)

seg_company = (
    defaults[["inn"]]
    .drop_duplicates()
    .merge(seg_zo, on="inn", how="left")
    .merge(seg_val4, on="inn", how="left")
)
seg_company["segment"] = np.where(seg_company["segment_zo"].isin(SEG_ORDER), seg_company["segment_zo"], seg_company["segment_val4"])

fp = (
    fp_raw.merge(seg_company[["inn", "segment"]], on="inn", how="left")
    .dropna(subset=["segment"])
    [["inn", "fp_sfp_code", "type_fp", "fp_date", "segment"]]
    .copy()
)
fp["fp_sfp_code"] = fp["fp_sfp_code"].astype(str).str.strip()
fp["type_fp"] = fp["type_fp"].astype(str).str.strip().replace({"": np.nan})

cohort = defaults.merge(seg_company[["inn", "segment"]], on="inn", how="left")
cohort = cohort[cohort["segment"].isin(SEG_ORDER)].copy()

print(f"Когорта с сегментами: {len(cohort):,}")
print(cohort["segment"].value_counts().reindex(SEG_ORDER, fill_value=0).to_string())
print(f"FP-записей после сегментации: {len(fp):,}")

In [ ]:
# База событий ФП относительно default_date
base = fp.merge(cohort[["inn", "default_date", "segment"]], on=["inn", "segment"], how="inner")
base["days_before_default"] = (base["default_date"] - base["fp_date"]).dt.days
base = base[base["days_before_default"] >= 0].copy()

print(f"Событий ФП до/в день дефолта: {len(base):,}")

In [ ]:
# Интервальная логика (взаимоисключающие промежутки)
rows_pool = []
rows_long = []

for interval_name, left_m, right_m in INTERVALS:
    left_dt = base["default_date"] - pd.DateOffset(months=left_m)
    right_dt = base["default_date"] - pd.DateOffset(months=right_m)

    # left inclusive, right exclusive; для последнего интервала 6_0m right inclusive
    if right_m == 0:
        mask = (base["fp_date"] >= left_dt) & (base["fp_date"] <= right_dt)
    else:
        mask = (base["fp_date"] >= left_dt) & (base["fp_date"] < right_dt)

    w = base[mask].copy()

    uniq = w[["inn", "segment", "fp_sfp_code", "type_fp"]].drop_duplicates()

    pool = (
        uniq.groupby(["inn", "segment"], as_index=False)
        .agg(fp_pool=("fp_sfp_code", lambda s: sorted(set(map(str, s)))))
    )

    pool_full = cohort[["inn", "segment"]].merge(pool, on=["inn", "segment"], how="left")
    pool_full["interval"] = interval_name
    pool_full["fp_pool"] = pool_full["fp_pool"].apply(lambda x: x if isinstance(x, list) else None)
    rows_pool.append(pool_full)

    uniq["interval"] = interval_name
    rows_long.append(uniq)

pool_by_inn_interval = pd.concat(rows_pool, ignore_index=True)
long_for_top = pd.concat(rows_long, ignore_index=True)

print("Пул по ИНН x интервал (превью):")
print(pool_by_inn_interval.head(10))
print(f"\nВсего строк в пуле: {len(pool_by_inn_interval):,}")

In [ ]:
# TOP-10 по сегментам и интервалам

denom = (
    pool_by_inn_interval
    .groupby(["interval", "segment"], as_index=False)
    .agg(total_defaults=("inn", "nunique"))
)

top_base = (
    long_for_top
    .groupby(["interval", "segment", "fp_sfp_code", "type_fp"], as_index=False)
    .agg(count_inn=("inn", "nunique"))
    .merge(denom, on=["interval", "segment"], how="left")
)

top_base["share_in_segment_defaults"] = top_base["count_inn"] / top_base["total_defaults"]

top_base = top_base.sort_values(
    ["interval", "segment", "count_inn", "share_in_segment_defaults", "fp_sfp_code"],
    ascending=[True, True, False, False, True]
)

top_base["rank"] = top_base.groupby(["interval", "segment"]).cumcount() + 1
top10_intervals_all = top_base[top_base["rank"] <= 10].copy()

print("TOP-10 по интервалам (превью):")
display(top10_intervals_all.head(40))

In [ ]:
# Контроль качества
expected_n = cohort["inn"].nunique()
check_pool = (
    pool_by_inn_interval
    .groupby("interval", as_index=False)
    .agg(n_inn=("inn", "nunique"), n_rows=("inn", "size"))
)
check_pool["expected_n"] = expected_n
check_pool["ok"] = (check_pool["n_inn"] == expected_n) & (check_pool["n_rows"] == expected_n)

none_stats = (
    pool_by_inn_interval
    .assign(is_none=lambda x: x["fp_pool"].isna())
    .groupby(["interval", "segment"], as_index=False)
    .agg(total_defaults=("inn", "nunique"), none_count=("is_none", "sum"))
)
none_stats["none_share"] = none_stats["none_count"] / none_stats["total_defaults"]

rank_dups = top10_intervals_all.duplicated(subset=["interval", "segment", "rank"]).sum()

print("Проверка полноты пула (ИНН по интервалам):")
display(check_pool)

print("\nСтатистика None по интервалам и сегментам:")
display(none_stats.sort_values(["interval", "segment"]))

print(f"\nДубликатов по ключу (interval, segment, rank): {rank_dups}")

In [ ]:
# Экспорт
out_xlsx = DATA_DIR / "Приложение_к_отчету_TOP10_FPSFP_по_интервалам_до_default_date.xlsx"

_engine = None
try:
    import openpyxl  # noqa: F401
    _engine = "openpyxl"
except Exception:
    try:
        import xlsxwriter  # noqa: F401
        _engine = "xlsxwriter"
    except Exception:
        _engine = None

_writer_kwargs = {"engine": _engine} if _engine else {}

with pd.ExcelWriter(out_xlsx, **_writer_kwargs) as writer:
    top10_intervals_all.to_excel(writer, sheet_name="top10_intervals_all", index=False)

    pool_export = pool_by_inn_interval.copy()
    pool_export["fp_pool"] = pool_export["fp_pool"].apply(lambda x: None if x is None else ", ".join(x))
    pool_export.to_excel(writer, sheet_name="pool_by_inn_interval", index=False)

    check_pool.to_excel(writer, sheet_name="check_pool_fullness", index=False)
    none_stats.to_excel(writer, sheet_name="none_stats", index=False)
    seg_company.to_excel(writer, sheet_name="segment_assignment", index=False)

    for interval_name, _, _ in INTERVALS:
        for seg in SEG_ORDER:
            sub = top10_intervals_all[(top10_intervals_all["interval"] == interval_name) & (top10_intervals_all["segment"] == seg)].copy()
            sheet = f"top10_{interval_name}_{seg}"[:31]
            sub.to_excel(writer, sheet_name=sheet, index=False)

print(f"Экспорт сохранен: {out_xlsx} (engine={_engine or 'auto'})")